# Eval: Finetuned vs Base Model — Retrieval Accuracy trên val.csv

Chạy được trên **Kaggle GPU (T4/P100)**. Không rebuild dual index — chỉ raw-text embedding để isolate effect của finetuning.

Pipeline: encode train → FAISS index → encode val → top-K majority vote → accuracy/F1

In [ ]:
!pip install -q sentence-transformers faiss-cpu einops

## 1. CONFIG — điền path vào đây

In [ ]:
from pathlib import Path

# ════════════════════════════════════════════════════════════
# DATA — điền path tới train.csv và val.csv của bạn
# ════════════════════════════════════════════════════════════
TRAIN_CSV       = 'data/split/essays/train.csv'
VAL_CSV         = 'data/split/essays/val.csv'

# ════════════════════════════════════════════════════════════
# MODEL — điền path tới finetuned model folder
# ════════════════════════════════════════════════════════════
FINETUNED_MODEL = 'models/sbert_essays_finetuned'
BASE_MODEL      = 'nomic-ai/nomic-embed-text-v1.5'

# ════════════════════════════════════════════════════════════
# EVAL CONFIG
# ════════════════════════════════════════════════════════════
TOP_K        = 5
ENCODE_BATCH = 16   # T4 16GB: 16 OK với seq=2048; giảm xuống 8 nếu OOM

# ════════════════════════════════════════════════════════════
TRAITS = ['cOPN', 'cCON', 'cEXT', 'cAGR', 'cNEU']
TRAIT_NAMES = {
    'cOPN': 'Openness', 'cCON': 'Conscientiousness',
    'cEXT': 'Extraversion', 'cAGR': 'Agreeableness', 'cNEU': 'Neuroticism',
}

# Path check
print('PATH CHECK')
for label, p in [('TRAIN_CSV', TRAIN_CSV), ('VAL_CSV', VAL_CSV), ('FINETUNED_MODEL', FINETUNED_MODEL)]:
    ok = 'OK' if Path(p).exists() else 'MISSING !!!'
    print(f'  {label:20s}: [{ok}]  {p}')

## 2. Imports & GPU check

In [ ]:
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
else:
    print('No GPU — CPU mode (chậm, ~40-60 phút/model)')
print(f'Device: {device}')

## 3. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

# Labels có thể là 'high'/'low' hoặc 0/1 tùy file
LABEL_MAP = {'high': 1, 'low': 0, 'y': 1, 'n': 0, 1: 1, 0: 0}
for t in TRAITS:
    train_df[t] = train_df[t].map(LABEL_MAP).fillna(train_df[t]).astype(int)
    val_df[t]   = val_df[t].map(LABEL_MAP).fillna(val_df[t]).astype(int)

print(f'Train: {len(train_df)} essays | Val: {len(val_df)} essays')
print(f'Label check — train cOPN: {train_df["cOPN"].value_counts().to_dict()}')

## 4. Helpers

In [ ]:
PREFIX = 'search_document: '

def encode_texts(model, texts, batch_size=ENCODE_BATCH):
    prefixed = [PREFIX + str(t) for t in texts]
    return model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)


def build_faiss(embs):
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index


def retrieval_accuracy(val_df, train_df, neighbor_idx):
    results = {}
    for trait in TRAITS:
        train_labels = train_df[trait].values
        val_labels   = val_df[trait].values
        preds = [int(train_labels[neighbor_idx[i]].mean() >= 0.5) for i in range(len(val_df))]
        results[trait] = {
            'acc': accuracy_score(val_labels, preds),
            'f1':  f1_score(val_labels, preds, average='macro'),
        }
    return results


print('Helpers OK')

## 5. Eval — Finetuned model

In [ ]:
print('Loading finetuned model...')
ft_model = SentenceTransformer(FINETUNED_MODEL, trust_remote_code=True, device=device)
print(f'  max_seq_length : {ft_model.max_seq_length}')
print(f'  embedding dim  : {ft_model.get_sentence_embedding_dimension()}')

print('\nEncoding train...')
ft_train_emb = encode_texts(ft_model, train_df['text'].tolist())
ft_index     = build_faiss(ft_train_emb)
print(f'Index built: {ft_index.ntotal} vectors')

print('\nEncoding val...')
ft_val_emb  = encode_texts(ft_model, val_df['text'].tolist())
_, ft_nbrs  = ft_index.search(ft_val_emb, TOP_K)
ft_results  = retrieval_accuracy(val_df, train_df, ft_nbrs)

del ft_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Done.')

## 6. Eval — Base model (comparison)

In [ ]:
print('Loading base model...')
base_model = SentenceTransformer(BASE_MODEL, trust_remote_code=True, device=device)
base_model.max_seq_length = 2048
print(f'  max_seq_length : {base_model.max_seq_length}')

print('\nEncoding train...')
base_train_emb = encode_texts(base_model, train_df['text'].tolist())
base_index     = build_faiss(base_train_emb)

print('\nEncoding val...')
base_val_emb   = encode_texts(base_model, val_df['text'].tolist())
_, base_nbrs   = base_index.search(base_val_emb, TOP_K)
base_results   = retrieval_accuracy(val_df, train_df, base_nbrs)

del base_model
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Done.')

## 7. Kết quả

In [ ]:
print(f'=== Retrieval accuracy (top-{TOP_K} majority vote, raw-text only) ===\n')
print(f'{"Trait":22s} {"Base Acc":>10s} {"FT Acc":>10s} {"Δ Acc":>8s}   {"Base F1":>8s} {"FT F1":>8s} {"Δ F1":>8s}')
print('-' * 82)

acc_deltas, f1_deltas = [], []
for trait in TRAITS:
    b  = base_results[trait]
    ft = ft_results[trait]
    da  = ft['acc'] - b['acc']
    df1 = ft['f1']  - b['f1']
    acc_deltas.append(da)
    f1_deltas.append(df1)
    arrow = '↑' if da > 0.005 else ('↓' if da < -0.005 else '~')
    print(f'  {TRAIT_NAMES[trait]:20s} {b["acc"]:>10.4f} {ft["acc"]:>10.4f} {da:>+8.4f} {arrow}  '
          f'{b["f1"]:>8.4f} {ft["f1"]:>8.4f} {df1:>+8.4f}')

print('-' * 82)
avg_b_acc  = np.mean([base_results[t]['acc'] for t in TRAITS])
avg_ft_acc = np.mean([ft_results[t]['acc']   for t in TRAITS])
avg_b_f1   = np.mean([base_results[t]['f1']  for t in TRAITS])
avg_ft_f1  = np.mean([ft_results[t]['f1']    for t in TRAITS])
print(f'  {"AVG":20s} {avg_b_acc:>10.4f} {avg_ft_acc:>10.4f} {avg_ft_acc-avg_b_acc:>+8.4f}    '
      f'{avg_b_f1:>8.4f} {avg_ft_f1:>8.4f} {avg_ft_f1-avg_b_f1:>+8.4f}')

print('\nΔ > 0 = finetuned tốt hơn base (raw-text only, không dual).')